Automação Simples Nacional

Esta Automação tem como foco o calculo do simples nacional o primeiro passo é juntar todas as planilhas contento as informações das notas em forma de csv.

In [ ]:
import pandas as pd
import glob
from selenium import webdriver
import time
import pytest
import pyautogui
import pyscreenshot
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select

#Selecionando todos os Arquivos csv pelo tipo podendo conter qualquer nome
csv_files = glob.glob('*.{}'.format('csv'))

csv_files

#Concatenando os arquivos
df_concat = pd.concat([pd.read_csv(f,sep=';',header=0,encoding='unicode-escape') for f in csv_files], ignore_index=True)
df_concat

#Juntando somente os dados que vão importar para o calculo do simples e sua conferencia
df_result = df_concat[['Nº da Nota Fiscal Eletrônica','Status da Nota Fiscal','Data da Emissão NFS-e/DSR-e','CPF/CNPJ do Prestador','Razão Social do Prestador','CPF/CNPJ do Tomador','Inscrição Municipal do Tomador','Inscrição Estadual do Tomador','Razão Social do Tomador','Código de Atividade Federal','Código de Atividade Municipal','Alíquota','Valor dos Serviços','Valor do ISS']]


#Adicionando novas colunas para realização do calculo do simples
df_result.loc[:, 'IRPJ'] = ''
df_result.loc[:, 'CSLL'] = ''
df_result.loc[:, 'Cofins'] = ''
df_result.loc[:, 'PIS/Pasep'] = ''
df_result.loc[:, 'CPP'] = ''
df_result.loc[:, 'ISS'] = ''
df_result.loc[:, 'ICMS'] = ''
df_result.loc[:, 'Total SN'] = ''
df_result.loc[:, 'Total retido'] = ''
df_result.loc[:, 'Total DAS'] = ''


#Obtendo CNPJ para verificação de quem é 
cnpj_input = input("Digite o CNPJ do cliente: ")

#Tratando CNPJ
cnpj_input_cleaned = ''.join(filter(str.isdigit, cnpj_input))

df_result['CPF/CNPJ do Tomador'] = df_result['CPF/CNPJ do Tomador'].astype(str).apply(lambda x: ''.join(filter(str.isdigit, x)))

selected_rows = df_result[df_result['CPF/CNPJ do Tomador'] == cnpj_input_cleaned]

#Verificando se não está vazio
if not selected_rows.empty:

    #Armazenando o valor do nome e CNPJ
    nome_do_cliente = selected_rows.iloc[0]['Razão Social do Tomador']
    print(f"Nome do cliente para CNPJ {cnpj_input}: {nome_do_cliente}")

    #Obtendo inscrição estadual para saber o anexo
    inscricao_estadual_values = selected_rows['Inscrição Municipal do Tomador'].unique()

    #Verificando se inscrição está no mesmo indice do cnpj e nome
    if any(inscricao_estadual_values):
        resultado_final = df_result[(df_result['CPF/CNPJ do Tomador'] == cnpj_input_cleaned) & (df_result['Razão Social do Tomador'] == nome_do_cliente) & (df_result['Inscrição Municipal do Tomador'].notna())]
        print(resultado_final)
    else:
        print("Nenhum valor encontrado na coluna 'Inscrição Estadual do Tomador'.")
else:
    print(f"Nenhum resultado encontrado para o CNPJ {cnpj_input}")
#df_result.to_excel("planilhaResultadoTeste.xlsx",index=False)
df_result


#Arryas para realização do Calculo completo do Simples Nacional
anexo = int(input("Digite o Anexo: "))
rbt12 = float(input("Digite o valor de RBT12: "))

aliquota_array = {
    1: {
        1: {"Aliquota": 4, "Desconto": 0},
        2: {"Aliquota": 7.3, "Desconto": 5940},
        3: {"Aliquota": 9.5, "Desconto": 13860},
        4: {"Aliquota": 10.7, "Desconto": 22500},
        5: {"Aliquota": 14.3, "Desconto": 87300},
        6: {"Aliquota": 19, "Desconto": 378000},
    },
    2: {
        1: {"Aliquota": 4.5, "Desconto": 0},
        2: {"Aliquota": 7.8, "Desconto": 5940},
        3: {"Aliquota": 10, "Desconto": 13860},
        4: {"Aliquota": 11.2, "Desconto": 22500},
        5: {"Aliquota": 14.7, "Desconto": 85500},
        6: {"Aliquota": 30, "Desconto": 720000},
    },
    3: {
        1: {"Aliquota": 6, "Desconto": 0},
        2: {"Aliquota": 11.2, "Desconto": 9360},
        3: {"Aliquota": 13.5, "Desconto": 17640},
        4: {"Aliquota": 16, "Desconto": 35640},
        5: {"Aliquota": 21, "Desconto": 125640},
        6: {"Aliquota": 33, "Desconto": 648000},
    },
    4: {
        1: {"Aliquota": 4.5, "Desconto": 0},
        2: {"Aliquota": 9, "Desconto": 8100},
        3: {"Aliquota": 10.2, "Desconto": 12420},
        4: {"Aliquota": 14, "Desconto": 39780},
        5: {"Aliquota": 22, "Desconto": 183780},
        6: {"Aliquota": 33, "Desconto": 828000},
    },
    5: {
        1: {"Aliquota": 15.5, "Desconto": 0},
        2: {"Aliquota": 18, "Desconto": 4500},
        3: {"Aliquota": 19.5, "Desconto": 9900},
        4: {"Aliquota": 20.5, "Desconto": 17100},
        5: {"Aliquota": 23, "Desconto": 62100},
        6: {"Aliquota": 30.5, "Desconto": 540000},
    },
}

impostos_array = {
    "1": {
        1: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1274,
            "PIS": 0.0276,
            "CPP": 0.415,
            "ICMS": 0.34,
            "IPI": 0.0,
            "ISS": 0.0
          },
        2: {
            "IRPJ":0.055,
            "CSLL":0.035,
            "CONFINS":0.1151,
            "PIS":0.0249 ,
            "CPP": 0.375,
            "IPI": 0.075,
            "ICMS": 0.32,
            "ISS": 0.0
        },
        3: {
            "IRPJ": 0.04,
            "CSLL": 0.035,
            "COFINS": 0.1282,
            "PIS": 0.0278,
            "CPP": 0.434,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.335
        },
        4: {
            "IRPJ": 0.188,
            "CSLL": 0.152,
            "COFINS": 0.1767,
            "PIS": 0.0383,
            "CPP": 0.0,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.445
        },
        5: {
            "IRPJ": 0.25,
            "CSLL": 0.15,
            "COFINS": 0.141,
            "PIS": 0.0305,
            "CPP": 0.2885,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.14
        }
        
    },
    "2": {
        1: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1274,
            "PIS": 0.0276,
            "CPP": 0.415,
            "ICMS": 0.34,
            "IPI": 0.0,
            "ISS": 0.0
        },
        2: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1151,
            "PIS": 0.0249,
            "CPP": 0.375,
            "IPI": 0.075,
            "ICMS": 0.32,
            "ISS": 0.0
        },
        3: {
            "IRPJ": 0.04,
            "CSLL": 0.035,
            "COFINS": 0.1405,
            "PIS": 0.0305,
            "CPP": 0.434,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.32
        },
        4: {
            "IRPJ": 0.198,
            "CSLL": 0.152,
            "COFINS": 0.2055,
            "PIS": 0.0445,
            "CPP": 0.0,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.4
        },
        5: {
            "IRPJ": 0.23,
            "CSLL": 0.15,
            "COFINS": 0.141,
            "PIS": 0.0305,
            "CPP": 0.2785,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.17
        }
    },
    "3":{
        1: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1274,
            "PIS": 0.0276,
            "CPP": 0.42,
            "ICMS": 0.335,
            "IPI": 0.0,
            "ISS": 0.0
        },
        2: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1151,
            "PIS": 0.0249,
            "CPP": 0.375,
            "IPI": 0.075,
            "ICMS": 0.32,
            "ISS": 0.0
        },
        3: {
            "IRPJ": 0.04,
            "CSLL": 0.035,
            "COFINS": 0.1364,
            "PIS": 0.0296,
            "CPP": 0.434,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.325
        },
        4: {
            "IRPJ": 0.208,
            "CSLL": 0.152,
            "COFINS": 0.1973,
            "PIS": 0.0427,
            "CPP": 0.0,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.4
        },
        5: {
            "IRPJ": 0.24,
            "CSLL": 0.15,
            "COFINS": 0.1492,
            "PIS": 0.0323,
            "CPP": 0.2385,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.19
        }
    },
    "4": {
        1: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1274,
            "PIS": 0.0276,
            "CPP": 0.42,
            "ICMS": 0.335,
            "IPI": 0.0,
            "ISS": 0.0
        },
        2: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1151,
            "PIS": 0.0249,
            "CPP": 0.375,
            "IPI": 0.075,
            "ICMS": 0.32,
            "ISS": 0.0
        },
        3: {
            "IRPJ": 0.04,
            "CSLL": 0.035,
            "COFINS": 0.1364,
            "PIS": 0.0296,
            "CPP": 0.434,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.325
        },
        4: {
            "IRPJ": 0.178,
            "CSLL": 0.192,
            "COFINS": 0.189,
            "PIS": 0.041,
            "CPP": 0.0,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.4
        },
        5: {
            "IRPJ": 0.21,
            "CSLL": 0.15,
            "COFINS": 0.1574,
            "PIS": 0.0341,
            "CPP": 0.2385,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.21
        }
    },
    "5": {
        1: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1274,
            "PIS": 0.0276,
            "CPP": 0.42,
            "ICMS": 0.335,
            "IPI": 0.0,
            "ISS": 0.0
        },
        2: {
            "IRPJ": 0.055,
            "CSLL": 0.035,
            "COFINS": 0.1151,
            "PIS": 0.0249,
            "CPP": 0.375,
            "IPI": 0.075,
            "ICMS": 0.32,
            "ISS": 0.0
        },
        3: {
            "IRPJ": 0.23,
            "CSLL": 0.15,
            "COFINS": 0.141,
            "PIS": 0.0305,
            "CPP": 0.2785,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.17
        },
        4: {
            "IRPJ": 0.188,
            "CSLL": 0.192,
            "COFINS": 0.1808,
            "PIS": 0.0392,
            "CPP": 0.0,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.4
        },
        5: {
            "IRPJ": 0.23,
            "CSLL": 0.125,
            "COFINS": 0.141,
            "PIS": 0.0305000000000001,
            "CPP": 0.2385,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.235
        }
    },
    "6": {
        1: {
            "IRPJ": 0.135,
            "CSLL": 0.1,
            "COFINS": 0.2827,
            "PIS": 0.0613,
            "CPP": 0.421,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.0
        },
        2: {
            "IRPJ": 0.085,
            "CSLL": 0.075,
            "COFINS": 0.2096,
            "PIS": 0.0454,
            "CPP": 0.235,
            "IPI": 0.35,
            "ICMS": 0.0,
            "ISS": 0.0
        },
        3: {
            "IRPJ": 0.35,
            "CSLL": 0.15,
            "COFINS": 0.1603,
            "PIS": 0.0347,
            "CPP": 0.305,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.0
        },
        4: {
            "IRPJ": 0.535,
            "CSLL": 0.215,
            "COFINS": 0.2055,
            "PIS": 0.0445,
            "CPP": 0.0,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.0
        },
        5: {
            "IRPJ": 0.35,
            "CSLL": 0.155,
            "COFINS": 0.1644,
            "PIS": 0.0356,
            "CPP": 0.295,
            "ICMS": 0.0,
            "IPI": 0.0,
            "ISS": 0.0
        }
    }
}

faixa = None

if rbt12 <= 180000:
    faixa = 1
elif 180000 < rbt12 <= 360000:
    faixa = 2
elif 360000 < rbt12 <= 720000:
    faixa = 3
elif 720000 < rbt12 <= 1800000:
    faixa = 4
elif 1800000 < rbt12 <= 3600000:
    faixa = 5
elif 3600000 < rbt12 <= 4800000:
    faixa = 6
    
aliquota = aliquota_array[anexo][faixa]['Aliquota']
desconto = aliquota_array[anexo][faixa]['Desconto']
print(aliquota)
print(desconto)
aliquota /= 100
print(aliquota)
aliq_efetiva = ((rbt12 * aliquota) - desconto) / rbt12


print(f"A Aliquota Efetiva da Empresa {nome_do_cliente} é: {aliq_efetiva}")


#Exportanto em Formato Excel
df_result.to_excel('PlanilhaResultado.xlsx',index=False)

df_result